Imports and Cuda connection:

In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import classification_report, confusion_matrix

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


Data collection and view:

In [8]:
import os
for root, dirs, files in os.walk('../Data'):
    print(root, '->', dirs[:6], f'({len(files)} files)')

../Data -> ['seg_test', 'seg_train'] (0 files)
../Data\seg_test -> ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street'] (0 files)
../Data\seg_test\buildings -> [] (437 files)
../Data\seg_test\forest -> [] (474 files)
../Data\seg_test\glacier -> [] (553 files)
../Data\seg_test\mountain -> [] (525 files)
../Data\seg_test\sea -> [] (510 files)
../Data\seg_test\street -> [] (501 files)
../Data\seg_train -> ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street'] (0 files)
../Data\seg_train\buildings -> [] (2191 files)
../Data\seg_train\forest -> [] (2271 files)
../Data\seg_train\glacier -> [] (2404 files)
../Data\seg_train\mountain -> [] (2512 files)
../Data\seg_train\sea -> [] (2274 files)
../Data\seg_train\street -> [] (2382 files)


Data preprocessing:

In [9]:
IMG_SIZE = 128

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

test_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [10]:
train_ds = datasets.ImageFolder('../Data/seg_train', transform=train_tf)
test_ds  = datasets.ImageFolder('../Data/seg_test',  transform=test_tf)

print(train_ds.classes)
print(train_ds.class_to_idx)
print(len(train_ds), len(test_ds))

['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
{'buildings': 0, 'forest': 1, 'glacier': 2, 'mountain': 3, 'sea': 4, 'street': 5}
14034 3000


In [11]:
batch_size = 64
train_l = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=True)
test_l  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

In [12]:
xb, yb = next(iter(train_l))
print(xb.shape, xb.dtype)
print(yb.shape, yb.dtype)
print(yb[:10])

torch.Size([64, 3, 128, 128]) torch.float32
torch.Size([64]) torch.int64
tensor([1, 5, 2, 0, 4, 1, 4, 1, 5, 1])
